# Qwen3 reviewer experiments

Select a CUDA GPU runtime and run the cells in order. Set `REPO_REF` to a published branch containing the BGE-M3 runner. Choose the Qwen3 model in the configuration cell. The runner uses BGE-M3 retrieval. The final cell prints the ZIP path for download from the file browser.

See `docs/reviewer_experiments.md` in the checkout for experiment methods and CLI options.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Tejaroop-Vasam/ML-ICL.git"
REPO_REF = "v2"
CHECKOUT = Path("ml-icl-checkout").resolve()
OUTPUT = Path("ml_icl_review_runs").resolve()
CACHE = Path("ml_icl_benchmark_data").resolve()

# Add CLI arguments to select a study. An empty list runs the full pilot.
# Example: ["--datasets", "imdb", "clapnq", "--samples", "imdb=100"]
# EXPERIMENT_ARGS = [
#     "--model", "Qwen/Qwen3-0.6B",
#     "--model", "Qwen/Qwen3-1.7B",
#     "--model", "Qwen/Qwen3-4B",
#     "--model", "Qwen/Qwen3-8B",
# ]
EXPERIMENT_ARGS = ["--model", "Qwen/Qwen3-4B"]


In [ ]:
if not CHECKOUT.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
         "--branch", REPO_REF, REPO_URL, str(CHECKOUT)],
        check=True,
    )
else:
    origin = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=CHECKOUT, text=True,
    ).strip()
    if origin != REPO_URL:
        raise ValueError(f"Choose a different checkout directory: {CHECKOUT}")
    subprocess.run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=CHECKOUT, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=CHECKOUT, check=True)

subprocess.run(["git", "sparse-checkout", "set", "src", "docs"], cwd=CHECKOUT, check=True)
if not (CHECKOUT / "requirements-review.txt").is_file():
    raise FileNotFoundError(
        f"Git ref {REPO_REF!r} is missing requirements-review.txt. "
        "Set REPO_REF to the branch containing the BGE-M3 update and rerun both cells."
    )
if not (CHECKOUT / "src" / "retrieval.py").is_file():
    raise FileNotFoundError(f"Git ref {REPO_REF!r} is missing the BGE-M3 runner. "
                            "Set REPO_REF to the branch containing this update.")


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements-review.txt"],
    cwd=CHECKOUT, check=True,
)


In [ ]:
import torch

if "--profile-only" not in EXPERIMENT_ARGS and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. In Colab, choose Runtime → Change runtime type → GPU, "
        "then rerun the setup cells and this cell."
    )

subprocess.run(
    [sys.executable, "-m", "src.review_experiments", "run",
     "--output-root", str(OUTPUT), "--cache-root", str(CACHE), *EXPERIMENT_ARGS],
    cwd=CHECKOUT, stderr=subprocess.STDOUT, check=True,
)
